# Personal Financial Agent — Evaluation & Synthetic Data Generation

This notebook demonstrates the full evaluation lifecycle for our Romanian Personal Financial Agent,
directly adapted from **AIE9 Sessions 9-10**.

## Structure
1. **Synthetic Data Generation** — Generate test questions from financial documents using RAGAS
2. **RAG Evaluation — Baseline** — Evaluate with naive top-k retrieval
3. **RAG Evaluation — Improved** — Add Cohere reranking and compare scores
4. **Agent Evaluation** — Test tool routing, topic adherence, MiFID II compliance

In [1]:
# Setup & Imports
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Jupyter

import os
import sys
import asyncio
import json
import shutil
import concurrent.futures
import pandas as pd
from IPython.display import display, HTML, Markdown

# Add app to path (works in Docker and locally)
sys.path.insert(0, '/app' if os.path.exists('/app/app') else os.path.abspath('..'))

from app.config import settings

# When running notebook locally while services are in Docker,
# override Docker hostnames with localhost
if not os.path.exists('/app/app'):
    settings.qdrant_host = 'localhost'
    settings.postgres_host = 'localhost'
    settings.database_url = settings.database_url.replace('@postgres:', '@localhost:')

from app.services.rag_service import rag_service
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print(f'OpenAI API Key: {settings.openai_api_key[:8]}...')
print(f'Qdrant: {settings.qdrant_host}:{settings.qdrant_port}')
print(f'Collection: {settings.qdrant_collection}')


OpenAI API Key: sk-proj-...
Qdrant: localhost:6333
Collection: financial_docs_ro


## 1. Synthetic Data Generation (SDG)

Using RAGAS `TestsetGenerator` to create synthetic question-answer pairs from our Romanian
financial documents. This follows the AIE9 Session 9 pattern.

The generator creates three types of questions:
- **Simple** — Single-fact retrieval questions
- **Multi-Context** — Questions requiring information from multiple chunks
- **Reasoning** — Questions requiring inference from retrieved information

In [2]:
# Load documents for SDG
from langchain_community.document_loaders import PyMuPDFLoader

# Target key financial-product PDFs explicitly (reproducible)
# Works in Docker (/app/documents) and locally (backend/documents)
_cwd = os.getcwd()
if os.path.exists('/app/documents'):
    _docs_base = '/app/documents'
elif os.path.exists(os.path.join(_cwd, 'documents')):
    _docs_base = os.path.join(_cwd, 'documents')
elif os.path.exists(os.path.join(_cwd, '..', 'documents')):
    _docs_base = os.path.abspath(os.path.join(_cwd, '..', 'documents'))
else:
    _docs_base = os.path.abspath(os.path.join(_cwd, 'backend', 'documents'))
pdf_files = [
    os.path.join(_docs_base, 'Ghid_TEZAUR_si_FIDELIS.pdf'),
    os.path.join(_docs_base, 'ghid_investitor_titluri_stat_ue_2019.pdf'),
]

documents = []
for pdf in pdf_files:
    loader = PyMuPDFLoader(pdf)
    documents.extend(loader.load())

print(f'Loaded {len(documents)} pages from {len(pdf_files)} PDF files')
for doc in documents[:3]:
    print(f'  - {doc.metadata.get("source", "unknown")}: {doc.page_content[:100]}...')


Loaded 9 pages from 1 PDF files
  - /Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/documents/Ghid_TEZAUR_si_FIDELIS.pdf: Ghid TEZAUR și FIDELIS
Ghid complet TEZAUR și FIDELIS – Titluri de stat
pentru populație
Document de...
  - /Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/documents/Ghid_TEZAUR_si_FIDELIS.pdf: Cine poate investi
Persoane fizice: cetățeni români care au împlinit 18 ani la data subscrierii.
La ...
  - /Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/documents/Ghid_TEZAUR_si_FIDELIS.pdf: Subscriere online Ghișeul.ro: din martie 2025, cetățenii români cu vârsta peste 18 ani din întreaga ...


In [3]:
# Generate synthetic test set
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.testset.transforms import (
    SummaryExtractor, EmbeddingExtractor, CosineSimilarityBuilder,
    OverlapScoreBuilder, CustomNodeFilter, Parallel,
)
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor

# Pre-chunk full PDF pages into smaller pieces for the KG
sdg_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
sdg_chunks = sdg_splitter.split_documents(documents)
print(f'Split {len(documents)} pages into {len(sdg_chunks)} chunks for SDG')

# Setup LLM and embeddings for SDG
generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o-mini', api_key=settings.openai_api_key))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(
    model=settings.embedding_model,
    api_key=settings.openai_api_key
))

# Explicit transforms that skip HeadlinesExtractor/HeadlineSplitter
# (mirrors the medium-doc branch from ragas default_transforms)
sdg_transforms = [
    SummaryExtractor(llm=generator_llm),
    CustomNodeFilter(llm=generator_llm),
    Parallel(
        EmbeddingExtractor(
            embedding_model=generator_embeddings,
            property_name="summary_embedding",
            embed_property_name="summary",
        ),
        ThemesExtractor(llm=generator_llm),
        NERExtractor(llm=generator_llm),
    ),
    Parallel(
        CosineSimilarityBuilder(
            property_name="summary_embedding",
            new_property_name="summary_similarity",
            threshold=0.5,
        ),
        OverlapScoreBuilder(threshold=0.01),
    ),
]

# Fresh generator instance
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

def _run_sdg():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    result = generator.generate_with_langchain_docs(
        documents=sdg_chunks,
        testset_size=10,
        transforms=sdg_transforms,
        raise_exceptions=False,
    )
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    testset = pool.submit(_run_sdg).result()

test_df = testset.to_pandas()
print(f'Generated {len(test_df)} synthetic test questions')
print(f'Columns: {list(test_df.columns)}')
display(test_df.head(15))


Split 9 pages into 19 chunks for SDG


Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/19 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/57 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

Generated 12 synthetic test questions
Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']


,user_input,reference_contexts,reference,synthesizer_name
0,How can individuals subscribe to the TEZAUR go...,[Ghid TEZAUR și FIDELIS\nGhid complet TEZAUR ș...,Individuals can subscribe to the TEZAUR govern...,single_hop_specifc_query_synthesizer
1,"Cine poate investi, cetățeni români?",[Cine poate investi\nPersoane fizice: cetățeni...,Cetățenii români care au împlinit 18 ani la da...,single_hop_specifc_query_synthesizer
2,Ce este Trezoreria și cum poate fi accesată de...,[Online (SPV): anaf.ro – Spațiul Privat Virtua...,Trezoreria se referă la Oficiile Trezoreriei S...,single_hop_specifc_query_synthesizer
3,Cum poate fi utilizat Spațiul Privat Virtual A...,[Subscriere online Ghișeul.ro: din martie 2025...,Spațiul Privat Virtual ANAF este una dintre pl...,single_hop_specifc_query_synthesizer
4,What are the subscription methods for governme...,[<1-hop>\n\nOnline (SPV): anaf.ro – Spațiul Pr...,Government bonds in EUR can only be subscribed...,multi_hop_abstract_query_synthesizer
5,Ce drepturi egale oferă titlurile în funcție d...,[<1-hop>\n\nFiecare titlu dă drepturi egale de...,"Fiecare titlu dă drepturi egale deținătorilor,...",multi_hop_abstract_query_synthesizer
6,Cum a influențat subscrierea online prin Ghișe...,[<1-hop>\n\nSubscriere online Ghișeul.ro: din ...,Subscrierea online prin Ghișeul.ro a avut un i...,multi_hop_abstract_query_synthesizer
7,Ce informații sunt necesare pentru a subscrie ...,[<1-hop>\n\nGhișeul.ro) cu act de identitate ș...,"Pentru a subscrie la titluri în EUR, este nece...",multi_hop_abstract_query_synthesizer
8,Ce informații sunt disponibile despre programu...,[<1-hop>\n\nGhișeul.ro) cu act de identitate ș...,Programul TEZAUR a fost lansat în 2018 și a av...,multi_hop_specific_query_synthesizer
9,What are the key details regarding the subscri...,[<1-hop>\n\n2022: Creștere puternică; dobânzi ...,The FIDELIS program has shown significant grow...,multi_hop_specific_query_synthesizer


In [4]:
# If SDG fails (e.g., not enough documents), use manually curated test questions
# This is a fallback that ensures the evaluation can always run

MANUAL_TEST_QUESTIONS = [
    {
        'user_input': 'Ce sunt titlurile de stat TEZAUR?',
        'reference': 'Titlurile TEZAUR sunt instrumente financiare emise de Ministerul Finantelor din Romania, destinate exclusiv persoanelor fizice rezidente. Au maturitati de 1, 3 sau 5 ani, dobanda fixa, si sunt 100% garantate de statul roman. Sunt scutite de impozit pe venit.',
    },
    {
        'user_input': 'Care sunt diferentele intre TEZAUR si FIDELIS?',
        'reference': 'TEZAUR nu se tranzactioneaza pe bursa si este scutit de impozit. FIDELIS este listat la BVB, poate fi tranzactionat pe piata secundara, si este impozitat cu 10% din 2023.',
    },
    {
        'user_input': 'Ce avantaje are TEZAUR fata de depozitele bancare?',
        'reference': 'Nu exista risc de pierdere a capitalului investit. Dobanzile sunt mai mari decat la depozitele bancare. Scutire de impozit pe venit. Accesibile de la 1 RON.',
    },
    {
        'user_input': 'Cum se pot achizitiona titlurile FIDELIS?',
        'reference': 'FIDELIS sunt listate la BVB si pot fi cumparate sau vandute pe piata secundara. Dobanda fixa, platita semestrial sub forma de cupon.',
    },
    {
        'user_input': 'Ce maturitati au titlurile de stat romanesti?',
        'reference': 'Titlurile TEZAUR si FIDELIS au maturitati de 1 an, 3 ani sau 5 ani. FIDELIS poate fi denominat in LEI sau EURO.',
    },
]

# Use SDG results if available, otherwise fall back to manual
try:
    if len(test_df) >= 5:
        # Auto-detect column names (RAGAS 0.2.x uses user_input/reference)
        q_col = 'user_input' if 'user_input' in test_df.columns else 'question'
        gt_col = 'reference' if 'reference' in test_df.columns else 'ground_truth'
        eval_questions = test_df[q_col].tolist()
        eval_ground_truths = test_df[gt_col].tolist()
        print(f'Using {len(eval_questions)} SDG-generated questions')
    else:
        raise ValueError('Not enough SDG questions')
except Exception:
    eval_questions = [q['user_input'] for q in MANUAL_TEST_QUESTIONS]
    eval_ground_truths = [q['reference'] for q in MANUAL_TEST_QUESTIONS]
    print(f'Using {len(eval_questions)} manually curated questions')

for i, q in enumerate(eval_questions, 1):
    print(f'{i}. {q}')


Using 12 SDG-generated questions
1. How can individuals subscribe to the TEZAUR government bonds through Ghișeul.ro, and what are the benefits of this method?
2. Cine poate investi, cetățeni români?
3. Ce este Trezoreria și cum poate fi accesată de investitorii de retail?
4. Cum poate fi utilizat Spațiul Privat Virtual ANAF pentru subscrierea titlurilor Tezaur?
5. What are the subscription methods for government bonds in EUR and how do they differ from those in RON?
6. Ce drepturi egale oferă titlurile în funcție de condițiile pieței?
7. Cum a influențat subscrierea online prin Ghișeul.ro creșterea economică a programului Tezaur în perioada 2025?
8. Ce informații sunt necesare pentru a subscrie la titluri în EUR și cum se compară cu procesul de subscriere prin Ghișeul.ro?
9. Ce informații sunt disponibile despre programul TEZAUR în perioada 2018-2026?
10. What are the key details regarding the subscription periods and investment distribution for the FIDELIS program as outlined on fidel

---

## 1.5 RAG Pipeline Walkthrough

Before evaluating, let's **demonstrate each retrieval technique** used in our pipeline.
This follows the same pattern as AIE9 Session 11 (Advanced Retrieval with LangChain).

Our pipeline combines **four** retrieval strategies:

| # | Technique | Purpose |
|---|---|---|
| 1 | **ParentDocumentRetriever** | Small-to-big: search child chunks, return parent context |
| 2 | **BM25Retriever** | Sparse keyword matching (exact terms like "TEZAUR", "BVB") |
| 3 | **EnsembleRetriever** | Fuses BM25 (30%) + Vector (70%) via Reciprocal Rank Fusion |
| 4 | **CohereRerank** | Cross-encoder reranking to filter top-N most relevant chunks |


### ParentDocumentRetriever (Small-to-Big)

A "small-to-big" strategy — the Parent Document Retriever works based on a simple principle:

1. We split the full document into large **parent** chunks (2000 chars).
2. Each parent chunk is further split into smaller **child** chunks (400 chars).
3. The child chunks are embedded and stored in **Qdrant** for similarity search.
4. The parent chunks are stored in an **in-memory DocStore**.
5. When we query, we match against child chunks but **return the parent** — giving the LLM more surrounding context.

This is critical for our dense regulatory PDFs where a single sentence's meaning depends on surrounding paragraphs.


In [5]:
# Show ParentDocumentRetriever configuration
print('=== ParentDocumentRetriever Configuration ===')
print(f'Parent chunk size:   {settings.rag_parent_chunk_size} chars (returned to LLM)')
print(f'Parent overlap:      {settings.rag_parent_chunk_overlap} chars')
print(f'Child chunk size:    {settings.rag_child_chunk_size} chars (used for embedding search)')
print(f'Child overlap:       {settings.rag_child_chunk_overlap} chars')
print(f'Embedding model:     {settings.embedding_model}')
print(f'Vector DB:           Qdrant @ {settings.qdrant_host}:{settings.qdrant_port}')
print(f'Collection:          {settings.qdrant_collection}')

# Show the splitters from rag_service
print(f'\nParent splitter:     {rag_service.parent_splitter}')
print(f'Child splitter:      {rag_service.child_splitter}')


=== ParentDocumentRetriever Configuration ===
Parent chunk size:   2000 chars (returned to LLM)
Parent overlap:      200 chars
Child chunk size:    400 chars (used for embedding search)
Child overlap:       50 chars
Embedding model:     text-embedding-3-small
Vector DB:           Qdrant @ localhost:6333
Collection:          financial_docs_ro

Parent splitter:     <langchain_text_splitters.character.RecursiveCharacterTextSplitter object at 0x1308cf560>
Child splitter:      <langchain_text_splitters.character.RecursiveCharacterTextSplitter object at 0x1308cf4d0>


### BM25 Retriever (Sparse Keyword Matching)

[BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on 
[Bag-of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) — a sparse representation
that compares documents based on shared terms and their frequencies.

**Why BM25 matters for Romanian financial documents:** Embedding models can miss exact
acronyms like "BVB", "ASF", "FIDELIS", or "MiFID II". BM25 catches these exact-match
queries that dense retrieval might overlook.


In [6]:
# Ingest target PDFs for evaluation
eval_docs_dir = '/tmp/eval_docs'
os.makedirs(eval_docs_dir, exist_ok=True)
for pdf in ['Ghid_TEZAUR_si_FIDELIS.pdf', 'ghidul_investitorului.pdf', 'ghid_investitor_titluri_stat_ue_2019.pdf']:
    src = f'{_docs_base}/{pdf}'
    dst = f'{eval_docs_dir}/{pdf}'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

def _run_ingest():
    loop = asyncio.new_event_loop()
    result = loop.run_until_complete(rag_service.ingest_documents(eval_docs_dir))
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    ingest_result = pool.submit(_run_ingest).result()
print(f'Ingestion: {ingest_result}')

# Copy BM25/docstore pickles to default path so rag_service.query() can find them
for pkl in ['bm25_retriever.pkl', 'docstore.pkl']:
    pkl_src = f'{eval_docs_dir}/{pkl}'
    pkl_dst = f'{_docs_base}/{pkl}'
    if os.path.exists(pkl_src):
        shutil.copy2(pkl_src, pkl_dst)

# Demonstrate BM25 vs Vector retrieval
test_query = 'Ce este FIDELIS si cum se tranzactioneaza pe BVB?'
print(f'\nQuery: "{test_query}"\n')

# BM25 retrieval
if rag_service.bm25_retriever:
    bm25_docs = rag_service.bm25_retriever.invoke(test_query)[:3]
    print('=== BM25 Results (keyword matching) ===')
    for j, doc in enumerate(bm25_docs, 1):
        source = doc.metadata.get('source', 'unknown').split('/')[-1]
        print(f'  [{j}] {source} (p.{doc.metadata.get("page", "?")}): {doc.page_content[:120]}...')
else:
    print('BM25 not initialized')

# Dense vector retrieval
def _run_vector_query():
    loop = asyncio.new_event_loop()
    result = loop.run_until_complete(rag_service.query(test_query, use_reranking=False))
    loop.close()
    return result

print('\n=== Vector Results (semantic similarity) ===')
with concurrent.futures.ThreadPoolExecutor() as pool:
    vector_docs = pool.submit(_run_vector_query).result()
for j, doc in enumerate(vector_docs[:3], 1):
    source = doc.metadata.get('source', 'unknown').split('/')[-1]
    print(f'  [{j}] {source} (p.{doc.metadata.get("page", "?")}): {doc.page_content[:120]}...')


Ingestion: {'documents_processed': 2, 'total_chunks': 358, 'collection': 'financial_docs_ro'}

Query: "Ce este FIDELIS si cum se tranzactioneaza pe BVB?"

=== BM25 Results (keyword matching) ===
  [1] Ghid_TEZAUR_si_FIDELIS.pdf (p.1): Ghid TEZAUR și FIDELIS
Ghid complet TEZAUR și FIDELIS – Titluri de stat
pentru populație
Document de sinteză bazat pe pr...
  [2] Ghid_TEZAUR_si_FIDELIS.pdf (p.8): Transferul se face pentru întreaga valoare subscrisă pe formular; nu se acceptă divizări parțiale care să încalce regula...
  [3] ghidul_investitorului.pdf (p.2): I. CE ESTE PIAȚA DE CAPITAL? 
4
1. Cine poate deveni investitor pe piața de capital din România? 
5
2. Care sunt tipuril...

=== Vector Results (semantic similarity) ===
  [1] Ghid_TEZAUR_si_FIDELIS.pdf (p.8): Transferul se face pentru întreaga valoare subscrisă pe formular; nu se acceptă divizări parțiale care să încalce regula...
  [2] Ghid_TEZAUR_si_FIDELIS.pdf (p.3): Subscriere online Ghișeul.ro: din martie 2025, cetățenii români 

### EnsembleRetriever (Hybrid Fusion)

The Ensemble Retriever combines 2 or more retrievers using 
[Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf).

Our configuration: **30% BM25 + 70% Vector** — this weights semantic understanding
higher but still benefits from exact keyword matching.


In [7]:
# Show the Ensemble Retriever configuration
print('=== EnsembleRetriever Configuration ===')
print(f'Retrievers:  BM25 + ParentDocumentRetriever (Vector)')
print(f'Weights:     [0.3, 0.7] (30% BM25, 70% Vector)')
print(f'Algorithm:   Reciprocal Rank Fusion')
print(f'Initial k:   {settings.rag_top_k} documents before reranking')

# Run ensemble retrieval
def _run_ensemble():
    loop = asyncio.new_event_loop()
    result = loop.run_until_complete(rag_service.query(test_query, use_reranking=False))
    loop.close()
    return result

print(f'\n=== Ensemble Results for: "{test_query}" ===')
with concurrent.futures.ThreadPoolExecutor() as pool:
    ensemble_docs = pool.submit(_run_ensemble).result()
for j, doc in enumerate(ensemble_docs[:5], 1):
    source = doc.metadata.get('source', 'unknown').split('/')[-1]
    print(f'  [{j}] {source} (p.{doc.metadata.get("page", "?")}): {doc.page_content[:120]}...')


=== EnsembleRetriever Configuration ===
Retrievers:  BM25 + ParentDocumentRetriever (Vector)
Weights:     [0.3, 0.7] (30% BM25, 70% Vector)
Algorithm:   Reciprocal Rank Fusion
Initial k:   10 documents before reranking

=== Ensemble Results for: "Ce este FIDELIS si cum se tranzactioneaza pe BVB?" ===
  [1] Ghid_TEZAUR_si_FIDELIS.pdf (p.8): Transferul se face pentru întreaga valoare subscrisă pe formular; nu se acceptă divizări parțiale care să încalce regula...
  [2] Ghid_TEZAUR_si_FIDELIS.pdf (p.3): Subscriere online Ghișeul.ro: din martie 2025, cetățenii români cu vârsta peste 18 ani din întreaga lume pot
subscrie ti...
  [3] ghidul_investitorului.pdf (p.14): II. INSTITUȚIILE PIEȚEI DE CAPITAL DIN ROMÂNIA
1. Bursa de Valori Bucuresti
a fost înfiinţată în 1995 ca instituţie de i...
  [4] ghidul_investitorului.pdf (p.14): reprezintă un indicator statistic care reflectă fluctuațiile pieței. BVB 
calculează și distribuie în timp real 7 indici...
  [5] Ghid_TEZAUR_si_FIDELIS.pdf (p.4): Ti

### CohereRerank (Contextual Compression)

The final quality gate — Cohere's `rerank-multilingual-v3.0` cross-encoder model
re-scores every candidate document against the query with deep understanding,
then selects the top-N most relevant. Unlike cosine similarity (which compares
embeddings), a cross-encoder sees the **full query and document together**.


In [8]:
# Run with reranking and compare
print(f'=== After Cohere Reranking for: "{test_query}" ===')
print(f'Reranker:     Cohere rerank-multilingual-v3.0')
print(f'Input docs:   {settings.rag_top_k} -> Output docs: {settings.rag_rerank_top_n}')
print()

def _run_reranked():
    loop = asyncio.new_event_loop()
    result = loop.run_until_complete(rag_service.query(test_query, use_reranking=True))
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    reranked_docs = pool.submit(_run_reranked).result()
for j, doc in enumerate(reranked_docs, 1):
    source = doc.metadata.get('source', 'unknown').split('/')[-1]
    relevance = doc.metadata.get('relevance_score', 'N/A')
    print(f'  [{j}] (score: {relevance}) {source} (p.{doc.metadata.get("page", "?")}): {doc.page_content[:120]}...')

# Summary
print(f'\n📊 Pipeline summary:')
print(f'  {len(documents)} PDF pages -> {settings.rag_parent_chunk_size}-char parent chunks -> {settings.rag_child_chunk_size}-char child chunks (embedded)')
print(f'  Query -> BM25 (30%) + Vector (70%) -> top-{settings.rag_top_k} -> Cohere Rerank -> top-{settings.rag_rerank_top_n} -> LLM')


=== After Cohere Reranking for: "Ce este FIDELIS si cum se tranzactioneaza pe BVB?" ===
Reranker:     Cohere rerank-multilingual-v3.0
Input docs:   10 -> Output docs: 5

  [1] (score: 0.9869795) Ghid_TEZAUR_si_FIDELIS.pdf (p.3): Subscriere online Ghișeul.ro: din martie 2025, cetățenii români cu vârsta peste 18 ani din întreaga lume pot
subscrie ti...
  [2] (score: 0.9827572) Ghid_TEZAUR_si_FIDELIS.pdf (p.8): Transferul se face pentru întreaga valoare subscrisă pe formular; nu se acceptă divizări parțiale care să încalce regula...
  [3] (score: 0.9014011) Ghid_TEZAUR_si_FIDELIS.pdf (p.1): Ghid TEZAUR și FIDELIS
Ghid complet TEZAUR și FIDELIS – Titluri de stat
pentru populație
Document de sinteză bazat pe pr...
  [4] (score: 0.258326) Ghid_TEZAUR_si_FIDELIS.pdf (p.4): Titluri în EUR:
3 ani: 3,50%
5 ani: 4,50%
10 ani: 6,00%
Unde se subscriu
Doar prin bănci și brokeri autorizați: BCR, BRD...
  [5] (score: 0.115560874) ghidul_investitorului.pdf (p.14): II. INSTITUȚIILE PIEȚEI DE CAPITAL DIN

## 2. RAG Evaluation — Baseline (Dense Vector Only)

We start with the simplest retrieval: **ParentDocumentRetriever** (dense vector similarity
search) with no BM25 fusion and no reranking. This is pure semantic search — our baseline.

In [9]:
# Run baseline RAG evaluation (no reranking)
from datasets import Dataset

def evaluate_rag(questions, ground_truths, use_reranking=False, use_ensemble=True):
    """Run RAG pipeline and collect results for RAGAS evaluation."""
    answers = []
    contexts = []
    llm = ChatOpenAI(model='gpt-4o-mini', api_key=settings.openai_api_key)

    async def _run():
        for i, question in enumerate(questions, 1):
            print(f'  [{i}/{len(questions)}] {question[:60]}...')
            if use_reranking and i > 1:
                await asyncio.sleep(7)  # Cohere Trial key: 10 calls/min
            docs = await rag_service.query(
                question, use_reranking=use_reranking, use_ensemble=use_ensemble,
            )
            context_texts = [doc.page_content for doc in docs]
            context_str = '\n\n'.join(context_texts)
            prompt = f'Based on the following context, answer the question.\n\nContext:\n{context_str}\n\nQuestion: {question}\n\nAnswer:'
            response = await llm.ainvoke(prompt)
            answers.append(response.content)
            contexts.append(context_texts)

    def _thread_target():
        loop = asyncio.new_event_loop()
        loop.run_until_complete(_run())
        loop.close()

    with concurrent.futures.ThreadPoolExecutor() as pool:
        pool.submit(_thread_target).result()

    return answers, contexts

print('Running baseline RAG evaluation (dense vector only)...')
baseline_answers, baseline_contexts = evaluate_rag(
    eval_questions, eval_ground_truths, use_reranking=False, use_ensemble=False,
)
print(f'Generated {len(baseline_answers)} answers')


Running baseline RAG evaluation (dense vector only)...
  [1/12] How can individuals subscribe to the TEZAUR government bonds...
  [2/12] Cine poate investi, cetățeni români?...
  [3/12] Ce este Trezoreria și cum poate fi accesată de investitorii ...
  [4/12] Cum poate fi utilizat Spațiul Privat Virtual ANAF pentru sub...
  [5/12] What are the subscription methods for government bonds in EU...
  [6/12] Ce drepturi egale oferă titlurile în funcție de condițiile p...
  [7/12] Cum a influențat subscrierea online prin Ghișeul.ro creștere...
  [8/12] Ce informații sunt necesare pentru a subscrie la titluri în ...
  [9/12] Ce informații sunt disponibile despre programul TEZAUR în pe...
  [10/12] What are the key details regarding the subscription periods ...
  [11/12] Cum se poate depune o cerere de subscriere la Trezorerie și ...
  [12/12] Ce avantaje fiscale oferă titlurile de stat FIDELIS și cum s...
Generated 12 answers


In [10]:
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# Create RAGAS dataset
baseline_dataset = Dataset.from_dict({
    'user_input': eval_questions,
    'response': baseline_answers,
    'retrieved_contexts': baseline_contexts,
    'reference': eval_ground_truths,
})

# Run RAGAS evaluation in a thread to avoid async deadlock
print('Running RAGAS metrics on baseline...')
def _run_ragas_baseline():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    result = ragas_evaluate(
        dataset=baseline_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    baseline_result = pool.submit(_run_ragas_baseline).result()

baseline_scores = {k: round(v, 4) for k, v in baseline_result._repr_dict.items()}
print('\n=== Tier 1: Dense Vector Scores ===')
for metric, score in baseline_scores.items():
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'  {metric:<25} {bar} {score:.4f}')


Running RAGAS metrics on baseline...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Task exception was never retrieved
future: <Task finished name='Task-906' coro=<AsyncClient.aclose() done, defined at /Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_client.py:2024> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.12/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_client.py", line 2031, in aclose
    await self._transport.aclose()
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 389, in aclose
    await self._pool.aclose()
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.v


=== Tier 1: Dense Vector Scores ===
  faithfulness              ██████████████████░░ 0.9050
  answer_relevancy          ██████████████████░░ 0.9308
  context_precision         █████████████████░░░ 0.8829
  context_recall            ██████████████████░░ 0.9250


## 3. RAG Evaluation — Improved (Hybrid Ensemble + Reranking)

We iterate twice on the baseline:

- **Tier 2 — Hybrid Ensemble**: Add BM25 keyword retrieval (30%) fused with vector (70%)
  via Reciprocal Rank Fusion + deduplication. No Cohere API calls needed.
- **Tier 3 — Hybrid + Rerank**: Same ensemble, plus Cohere `rerank-multilingual-v3.0`
  cross-encoder reranking for maximum precision.

This is the **iteration story** — we show measurable, incremental improvement.

In [11]:
# --- Tier 2: Hybrid Ensemble (BM25 + Vector, no reranking) ---
print('Running Tier 2 — Hybrid Ensemble (BM25 + Vector)...')
hybrid_answers, hybrid_contexts = evaluate_rag(
    eval_questions, eval_ground_truths, use_reranking=False, use_ensemble=True,
)

hybrid_dataset = Dataset.from_dict({
    'user_input': eval_questions,
    'response': hybrid_answers,
    'retrieved_contexts': hybrid_contexts,
    'reference': eval_ground_truths,
})

print('Running RAGAS metrics on hybrid pipeline...')
def _run_ragas_hybrid():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    result = ragas_evaluate(
        dataset=hybrid_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    hybrid_result = pool.submit(_run_ragas_hybrid).result()

hybrid_scores = {k: round(v, 4) for k, v in hybrid_result._repr_dict.items()}
print('\n=== Tier 2: Hybrid Ensemble Scores ===')
for metric, score in hybrid_scores.items():
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'  {metric:<25} {bar} {score:.4f}')

# --- Tier 3: Hybrid Ensemble + Cohere Rerank ---
import time
print('\nCooling down 7s for Cohere rate limit...')
time.sleep(7)
print('Running Tier 3 — Hybrid Ensemble + Cohere Rerank...')
reranked_answers, reranked_contexts = evaluate_rag(
    eval_questions, eval_ground_truths, use_reranking=True, use_ensemble=True,
)

reranked_dataset = Dataset.from_dict({
    'user_input': eval_questions,
    'response': reranked_answers,
    'retrieved_contexts': reranked_contexts,
    'reference': eval_ground_truths,
})

print('Running RAGAS metrics on reranked pipeline...')
def _run_ragas_reranked():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    result = ragas_evaluate(
        dataset=reranked_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    loop.close()
    return result

with concurrent.futures.ThreadPoolExecutor() as pool:
    reranked_result = pool.submit(_run_ragas_reranked).result()

reranked_scores = {k: round(v, 4) for k, v in reranked_result._repr_dict.items()}
print('\n=== Tier 3: Hybrid + Rerank Scores ===')
for metric, score in reranked_scores.items():
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'  {metric:<25} {bar} {score:.4f}')


Running Tier 2 — Hybrid Ensemble (BM25 + Vector)...
  [1/12] How can individuals subscribe to the TEZAUR government bonds...
  [2/12] Cine poate investi, cetățeni români?...
  [3/12] Ce este Trezoreria și cum poate fi accesată de investitorii ...
  [4/12] Cum poate fi utilizat Spațiul Privat Virtual ANAF pentru sub...
  [5/12] What are the subscription methods for government bonds in EU...
  [6/12] Ce drepturi egale oferă titlurile în funcție de condițiile p...
  [7/12] Cum a influențat subscrierea online prin Ghișeul.ro creștere...
  [8/12] Ce informații sunt necesare pentru a subscrie la titluri în ...
  [9/12] Ce informații sunt disponibile despre programul TEZAUR în pe...
  [10/12] What are the key details regarding the subscription periods ...
  [11/12] Cum se poate depune o cerere de subscriere la Trezorerie și ...
  [12/12] Ce avantaje fiscale oferă titlurile de stat FIDELIS și cum s...
Running RAGAS metrics on hybrid pipeline...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]


=== Tier 2: Hybrid Ensemble Scores ===
  faithfulness              █████████████████░░░ 0.8990
  answer_relevancy          ██████████████████░░ 0.9331
  context_precision         ████████████████░░░░ 0.8334
  context_recall            ██████████████████░░ 0.9250

Cooling down 7s for Cohere rate limit...
Running Tier 3 — Hybrid Ensemble + Cohere Rerank...
  [1/12] How can individuals subscribe to the TEZAUR government bonds...
  [2/12] Cine poate investi, cetățeni români?...
  [3/12] Ce este Trezoreria și cum poate fi accesată de investitorii ...
  [4/12] Cum poate fi utilizat Spațiul Privat Virtual ANAF pentru sub...
  [5/12] What are the subscription methods for government bonds in EU...
  [6/12] Ce drepturi egale oferă titlurile în funcție de condițiile p...
  [7/12] Cum a influențat subscrierea online prin Ghișeul.ro creștere...
  [8/12] Ce informații sunt necesare pentru a subscrie la titluri în ...
  [9/12] Ce informații sunt disponibile despre programul TEZAUR în pe...
  [10/12]

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Task exception was never retrieved
future: <Task finished name='Task-4490' coro=<AsyncClient.aclose() done, defined at /Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_client.py:2024> exception=RuntimeError('Event loop is closed')>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.12/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_client.py", line 2031, in aclose
    await self._transport.aclose()
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 389, in aclose
    await self._pool.aclose()
  File "/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.


=== Tier 3: Hybrid + Rerank Scores ===
  faithfulness              █████████████████░░░ 0.8575
  answer_relevancy          ████████████████░░░░ 0.8487
  context_precision         █████████████████░░░ 0.8962
  context_recall            ███████████████░░░░░ 0.7514


In [12]:
# Three-tier comparison
HEADER = (
    f'{"Metric":<25} {"Dense Vec":>10} {"Hybrid":>10} {"H+Rerank":>10} '
    f'{"Δ H-D":>8} {"Δ R-H":>8}'
)
print('\n' + '='*80)
print('COMPARISON: Dense Vector vs Hybrid Ensemble vs Hybrid+Rerank')
print('='*80)
print(HEADER)
print('-'*80)

comparison_data = []
for metric in baseline_scores:
    d = baseline_scores.get(metric, 0)
    h = hybrid_scores.get(metric, 0)
    r = reranked_scores.get(metric, 0)
    dh = h - d
    dr = r - h
    dh_s = f'+{dh:.4f}' if dh >= 0 else f'{dh:.4f}'
    dr_s = f'+{dr:.4f}' if dr >= 0 else f'{dr:.4f}'
    print(f'{metric:<25} {d:>10.4f} {h:>10.4f} {r:>10.4f} {dh_s:>8} {dr_s:>8}')
    comparison_data.append({
        'Metric': metric,
        'Dense Vector': d,
        'Hybrid Ensemble': h,
        'Hybrid+Rerank': r,
        'Delta (H-D)': dh,
        'Delta (R-H)': dr,
    })

print('\n')
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.style.format({
    'Dense Vector': '{:.4f}',
    'Hybrid Ensemble': '{:.4f}',
    'Hybrid+Rerank': '{:.4f}',
    'Delta (H-D)': '{:+.4f}',
    'Delta (R-H)': '{:+.4f}',
}))



COMPARISON: Dense Vector vs Hybrid Ensemble vs Hybrid+Rerank
Metric                     Dense Vec     Hybrid   H+Rerank    Δ H-D    Δ R-H
--------------------------------------------------------------------------------
faithfulness                  0.9050     0.8990     0.8575  -0.0060  -0.0415
answer_relevancy              0.9308     0.9331     0.8487  +0.0023  -0.0844
context_precision             0.8829     0.8334     0.8962  -0.0495  +0.0628
context_recall                0.9250     0.9250     0.7514  +0.0000  -0.1736




,Metric,Dense Vector,Hybrid Ensemble,Hybrid+Rerank,Delta (H-D),Delta (R-H)
0,faithfulness,0.9050,0.8990,0.8575,-0.0060,-0.0415
1,answer_relevancy,0.9308,0.9331,0.8487,+0.0023,-0.0844
2,context_precision,0.8829,0.8334,0.8962,-0.0495,+0.0628
3,context_recall,0.9250,0.9250,0.7514,+0.0000,-0.1736


## 4. Agent Evaluation

We evaluate the full LangGraph Supervisor agent on **six scenarios** covering all five
tools plus an off-topic guardrail:

| # | Scenario | Expected Tool | Tests |
|---|---|---|---|
| 1 | RAG Query (RO) | `rag_query` | Answer quality, MiFID II disclaimer |
| 2 | Market Search | `market_search` | Tool routing, live data relevance |
| 3 | Goals Query | `goals_summary` | Tool routing, returns seeded goal data |
| 4 | Create Goal | `create_goal` | Tool routing, goal confirmation |
| 5 | RAG Query (EN) | `rag_query` | Language detection, MiFID II in English |
| 6 | Off-Topic Guardrail | None | Refuses non-financial questions |

**Scoring (per scenario):**
- **Tool Call Accuracy (35%)** — Deterministic: did the agent route to the expected tool?
- **Answer Quality (35%)** — **LLM-as-judge** (GPT-4o-mini): scores the response 1–5 against
  a grading rubric defined per scenario, then normalized to 0–1. This replaces brittle keyword
  matching with robust semantic evaluation.
- **MiFID II Compliance (30%)** — Deterministic: disclaimer present when required, absent when not?

In [15]:
# --- Agent Setup ---
from app.services.agent_service import agent_service
from langchain_core.messages import AIMessage

try:
    await agent_service.setup()
    print('Agent service initialized')
except Exception as e:
    raise RuntimeError(
        f'Agent setup failed: {e}\n'
        'Make sure Postgres is running: docker compose up -d postgres'
    )

# Seed demo data: creates demo user + 3 goals if they don't already exist
from seed_demo_data import seed, DEMO_USER_ID
await seed()

# --- Helpers ---

DISCLAIMER_KEYWORDS = [
    'mifid', 'recomandare de investiții', 'consilier financiar',
    'investment advice', 'investment recommendation', 'financial advisor',
]

def extract_tools_used(messages):
    """Return list of tool names invoked during the agent turn."""
    tools = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tools.extend(tc['name'] for tc in msg.tool_calls)
    return tools

def has_mifid_disclaimer(text):
    lower = text.lower()
    return any(kw in lower for kw in DISCLAIMER_KEYWORDS)

# LLM-as-judge: scores answer quality 1-5 against a per-scenario rubric
judge_llm = ChatOpenAI(model='gpt-4o-mini', api_key=settings.openai_api_key, temperature=0)

JUDGE_PROMPT = """You are an evaluation judge for a Romanian Personal Financial Agent.

Score the agent's answer on a scale of 1-5 based on the grading criteria below.

**Grading criteria:**
{criteria}

**User question:**
{question}

**Agent answer:**
{answer}

Return ONLY a JSON object with two keys:
- "score": integer 1-5 (1=completely wrong, 2=poor, 3=acceptable, 4=good, 5=excellent)
- "reason": one sentence explaining the score
"""

async def llm_judge(question: str, answer: str, criteria: str) -> tuple[int, str]:
    """Ask GPT-4o-mini to score the answer 1-5 against the criteria."""
    prompt = JUDGE_PROMPT.format(criteria=criteria, question=question, answer=answer)
    resp = await judge_llm.ainvoke(prompt)
    text = resp.content.strip()
    if text.startswith('```'):
        text = text.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
    try:
        result = json.loads(text)
        return int(result['score']), result.get('reason', '')
    except Exception:
        return 3, f'Could not parse judge response: {resp.content[:100]}'

# --- Expanded Test Scenarios (with grading criteria for LLM-as-judge) ---

AGENT_TEST_SCENARIOS = [
    {
        'category': 'RAG Query (RO)',
        'message': 'Ce este TEZAUR?',
        'expected_tool': 'rag_query',
        'should_have_disclaimer': True,
        'grading_criteria': (
            'The response should explain what TEZAUR is (titluri de stat / government bonds '
            'for Romanian individuals), mention key characteristics like state guarantee or '
            'fixed interest rates, and be entirely in Romanian.'
        ),
    },
    {
        'category': 'Market Search',
        'message': 'Care este cursul EUR/RON astazi?',
        'expected_tool': 'market_search',
        'should_have_disclaimer': False,
        'grading_criteria': (
            'The response should provide a current or recent EUR/RON exchange rate with '
            'a numeric value, or clearly state that it searched for this data. '
            'Must be in Romanian.'
        ),
    },
    {
        'category': 'Goals Query',
        'message': 'Care sunt obiectivele mele financiare?',
        'expected_tool': 'goals_summary',
        'should_have_disclaimer': False,
        'grading_criteria': (
            'The response should list the user\'s financial goals with amounts in RON and '
            'progress information. It should mention at least one of the seeded goals '
            '(e.g., Mașină, Vacanță Grecia, or Fond de urgență).'
        ),
    },
    {
        'category': 'Create Goal',
        'message': 'Creează-mi un obiectiv financiar de 10000 RON pentru un laptop nou, cu contribuție lunară de 500 RON',
        'expected_tool': 'create_goal',
        'should_have_disclaimer': False,
        'grading_criteria': (
            'The response should confirm that a new financial goal was created for a laptop '
            'with a target of 10,000 RON. Should acknowledge the goal name and amount.'
        ),
    },
    {
        'category': 'RAG Query (EN)',
        'message': 'What are the differences between TEZAUR and FIDELIS?',
        'expected_tool': 'rag_query',
        'should_have_disclaimer': True,
        'grading_criteria': (
            'The response should explain differences between TEZAUR and FIDELIS (e.g., '
            'tradability on BVB, tax treatment, subscription methods). '
            'The entire response MUST be in English, not Romanian.'
        ),
    },
    {
        'category': 'Off-Topic Guardrail',
        'message': 'Care e rețeta de sarmale?',
        'expected_tool': None,
        'should_have_disclaimer': False,
        'grading_criteria': (
            'The agent should politely decline to answer this cooking question and redirect '
            'the user to financial topics. It must NOT provide an actual recipe or cooking '
            'instructions. Mentioning that it is a financial assistant is ideal.'
        ),
    },
]

# --- Run Evaluation ---

agent_results = []
for i, scenario in enumerate(AGENT_TEST_SCENARIOS, 1):
    print(f'\n--- Scenario {i}: {scenario["category"]} ---')
    print(f'Message: {scenario["message"]}')

    input_msgs, config = await agent_service._prepare_turn(
        message=scenario['message'],
        user_id=str(DEMO_USER_ID),
        session_id=f'eval-agent-{i}',
    )
    response = await agent_service.graph.ainvoke(input_msgs, config=config)
    all_messages = response['messages']

    ai_messages = [m for m in all_messages if isinstance(m, AIMessage) and m.content]
    answer = ai_messages[-1].content if ai_messages else ''

    # 1. Tool Call Accuracy (35%) — deterministic
    tools_used = extract_tools_used(all_messages)
    expected = scenario['expected_tool']
    if expected is None:
        tool_correct = len(tools_used) == 0
    else:
        tool_correct = expected in tools_used

    # 2. Answer Quality (35%) — LLM-as-judge
    judge_score, judge_reason = await llm_judge(
        scenario['message'], answer, scenario['grading_criteria']
    )
    quality_score = (judge_score - 1) / 4.0  # normalize 1-5 → 0.0-1.0

    # 3. MiFID II Compliance (30%) — deterministic
    disclaimer_present = has_mifid_disclaimer(answer)
    disclaimer_ok = disclaimer_present == scenario['should_have_disclaimer']

    # Weighted overall
    overall = (
        (1.0 if tool_correct else 0.0) * 0.35
        + quality_score * 0.35
        + (1.0 if disclaimer_ok else 0.0) * 0.30
    )

    agent_results.append({
        'Category': scenario['category'],
        'Tool Correct': '✅' if tool_correct else '❌',
        'Tools Used': ', '.join(tools_used) or 'none',
        'Quality': f'{judge_score}/5',
        'Judge Reason': judge_reason,
        'Disclaimer OK': '✅' if disclaimer_ok else '❌',
        'Overall': f'{overall:.2f}',
        'Response Preview': answer[:150] + '...',
    })
    print(f'  Tool: {"✅" if tool_correct else "❌"} ({", ".join(tools_used) or "none"})')
    print(f'  Quality: {judge_score}/5 — {judge_reason}')
    print(f'  Disclaimer: {"✅" if disclaimer_ok else "❌"} | Overall: {overall:.2f}')
    print(f'  Response: {answer[:150]}...')

print('\n\n=== Agent Evaluation Summary ===')
agent_df = pd.DataFrame(agent_results)
display(agent_df[['Category', 'Tool Correct', 'Tools Used', 'Quality', 'Disclaimer OK', 'Overall']])

print('\n=== LLM-as-Judge Reasoning ===')
for _, row in agent_df.iterrows():
    print(f'  {row["Category"]}: {row["Judge Reason"]}')


/Users/adrianbarcan/Projects/AIE9-Certification-Challenge/backend/.venv/lib/python3.12/site-packages/psycopg_pool/pool_async.py:163: RuntimeWarning: opening the async pool AsyncConnectionPool in the constructor is deprecated and will not be supported anymore in a future release. Please use `await pool.open()`, or use the pool as context manager using: `async with AsyncConnectionPool(...) as pool: `...
  warnings.warn(


Agent service initialized
Demo data already exists. Skipping.

--- Scenario 1: RAG Query (RO) ---
Message: Ce este TEZAUR?
  Tool: ✅ (rag_query)
  Quality: 4/5 — Răspunsul explică bine ce este TEZAUR și menționează caracteristici importante, dar nu face referire explicită la garanția de stat sau la ratele fixe ale dobânzii.
  Disclaimer: ✅ | Overall: 0.91
  Response: TEZAUR este un program de titluri de stat emis de Ministerul Finanțelor din România, destinat exclusiv populației. Aceste titluri sunt în formă demate...

--- Scenario 2: Market Search ---
Message: Care este cursul EUR/RON astazi?
  Tool: ✅ (market_search, market_search, market_search, market_search, market_search, market_search)
  Quality: 5/5 — Agentul a furnizat un curs EUR/RON actualizat cu un număr precis și a inclus surse pentru verificare.
  Disclaimer: ✅ | Overall: 1.00
  Response: Astăzi, 1 martie 2026, cursul oficial BNR pentru 1 Euro (EUR) este de aproximativ 5.0968 Lei (RON).

Surse:
1. [Valutare.ro](https://ww

,Category,Tool Correct,Tools Used,Quality,Disclaimer OK,Overall
0,RAG Query (RO),✅,rag_query,4/5,✅,0.91
1,Market Search,✅,"market_search, market_search, market_search, m...",5/5,✅,1.00
2,Goals Query,✅,"goals_summary, goals_summary, goals_summary, g...",5/5,✅,1.00
3,Create Goal,✅,"create_goal, create_goal",5/5,✅,1.00
4,RAG Query (EN),✅,rag_query,5/5,✅,1.00
5,Off-Topic Guardrail,✅,none,5/5,✅,1.00



=== LLM-as-Judge Reasoning ===
  RAG Query (RO): Răspunsul explică bine ce este TEZAUR și menționează caracteristici importante, dar nu face referire explicită la garanția de stat sau la ratele fixe ale dobânzii.
  Market Search: Agentul a furnizat un curs EUR/RON actualizat cu un număr precis și a inclus surse pentru verificare.
  Goals Query: The agent's answer clearly lists the user's financial goals with specific amounts in RON and progress information, including the seeded goal of 'Mașină'.
  Create Goal: The agent's response clearly confirms the creation of a financial goal for a laptop with the specified amount of 10,000 RON and acknowledges the monthly contribution of 500 RON.
  RAG Query (EN): The agent provided a comprehensive and clear comparison of TEZAUR and FIDELIS, covering key differences such as currency, subscription methods, investment values, early redemption, secondary market trading, and tax treatment.
  Off-Topic Guardrail: The agent politely declined to answer 

In [16]:
# Final summary
print('='*60)
print('EVALUATION COMPLETE')
print('='*60)

print(f'\nRAG Dense Vector:     {baseline_scores}')
print(f'RAG Hybrid Ensemble:  {hybrid_scores}')
print(f'RAG Hybrid+Rerank:    {reranked_scores}')

agent_pass = sum(1 for r in agent_results if float(r['Overall']) >= 0.7)
agent_total = len(agent_results)
print(f'\nAgent Scenarios:      {agent_total} tested')
print(f'Agent Pass Rate:      {agent_pass}/{agent_total} ({agent_pass/agent_total:.0%})')

tool_correct_count = sum(1 for r in agent_results if r['Tool Correct'] == '✅')
print(f'Tool Call Accuracy:   {tool_correct_count}/{agent_total} ({tool_correct_count/agent_total:.0%})')

quality_scores = [int(r['Quality'].split('/')[0]) for r in agent_results]
avg_quality = sum(quality_scores) / len(quality_scores)
print(f'Avg Answer Quality:   {avg_quality:.1f}/5 (LLM-as-judge)')

disclaimer_ok_count = sum(1 for r in agent_results if r['Disclaimer OK'] == '✅')
print(f'MiFID II Compliance:  {disclaimer_ok_count}/{agent_total} ({disclaimer_ok_count/agent_total:.0%})')


EVALUATION COMPLETE

RAG Dense Vector:     {'faithfulness': 0.905, 'answer_relevancy': 0.9308, 'context_precision': 0.8829, 'context_recall': 0.925}
RAG Hybrid Ensemble:  {'faithfulness': 0.899, 'answer_relevancy': 0.9331, 'context_precision': 0.8334, 'context_recall': 0.925}
RAG Hybrid+Rerank:    {'faithfulness': 0.8575, 'answer_relevancy': 0.8487, 'context_precision': 0.8962, 'context_recall': 0.7514}

Agent Scenarios:      6 tested
Agent Pass Rate:      6/6 (100%)
Tool Call Accuracy:   6/6 (100%)
Avg Answer Quality:   4.8/5 (LLM-as-judge)
MiFID II Compliance:  6/6 (100%)
